# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


## Colour palette — used in all charts (Q1–Q7)

In [2]:
# ── Colour palette for ALL charts (Q1–Q7) — CVD-safe, no red/green ─────────
ORANGE = '#E07B39'    # main accent = Africa / "the story"
BLUE   = '#2E75B6'    # second accent (Asia, growth)
YELLOW = '#F5C842'    # Europe
TEAL   = '#009E73'    # North America
PURPLE = '#CC79A7'    # South America
SKY    = '#56B4E9'    # Oceania

# Greys for context
GREY          = '#BBBBBB'
LIGHT_GREY    = '#DDDDDD'
LIGHTEST_GREY = '#EEEEEE'
TEXT_GREY     = '#777777'   

continent_colours = {
    'Africa':        ORANGE,
    'Asia':          BLUE,
    'Europe':        YELLOW,
    'North America': TEAL,
    'South America': PURPLE,
    'Oceania':       SKY,
}

# Diverging scale (Q5): orange = shrank, white = no change, blue = grew
diverging_scale = [ORANGE, '#FFFFFF', BLUE]

FONT = dict(family='Arial', size=12)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a line chart because it is the best chart for showing change and trends over the years(time). Each continent’s population is divided by its own 1970 value, so every line starts at 1 in 1970. That lets continents of very different sizes be compared directly.
The y-axis shows how many times each continent has grown since 1970. The fastest-growing continent, Africa, is the only line in orange, and all the others are grey, so we could know the answer from the begining 

In [ ]:
# Q1 — Line chart: growth relative to 1970 (index, 1970 = 1)
cont = df_long.groupby(['Continent', 'Year'])['Population'].sum().reset_index()
cont['Index'] = cont['Population'] / cont.groupby('Continent')['Population'].transform('first')   # first = 1970

last = cont.loc[cont['Year'] == 2022].sort_values('Index', ascending=False)
fastest = last['Continent'].iloc[0]

colour_map = {c: (ORANGE if c == fastest else GREY) for c in cont['Continent'].unique()}

fig = px.line(
    cont, x='Year', y='Index', color='Continent',
    color_discrete_map=colour_map,
    markers=True,
    labels={'Index': 'Population relative to 1970', 'Year': ''},
    template='plotly_white', height=500, width=950,
)
fig.update_traces(line=dict(width=2), marker=dict(size=5))
fig.update_traces(line=dict(width=4), selector=dict(name=fastest))

label_y, prev = [], None
for v in last['Index']:                       
    y = v if prev is None else min(v, prev - 0.13)
    label_y.append(y); prev = y
last['Label_y'] = label_y

for _, row in last.iterrows():
    fig.add_annotation(
        x=2022, y=row['Label_y'], text=f"{row['Continent']}  {row['Index']:.1f}×",
        showarrow=False, xanchor='left', xshift=8,
        font=dict(color=ORANGE if row['Continent'] == fastest else TEXT_GREY, size=12),
    )

fig.update_layout(
    title=f"{fastest} is growing fastest — its population is {last['Index'].iloc[0]:.1f}× its 1970 size",
    font=FONT, showlegend=False,
    xaxis=dict(showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2022]),
    yaxis=dict(gridcolor=LIGHTEST_GREY, ticksuffix='×'),
    margin=dict(r=170),
)
fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Sorted horizontal bar. I chose this type of bar because the question is about rating, and the length of bars from same zero line is easier to compare. 
African countries are orange and all others grey, so you can count the African countries at a glance, and the title gives the answer (4 of 15). Each bar is labelled with its value in millions, so the x-axis and legend could be removed.

In [ ]:
# Q2 — Sorted horizontal bar: top 15 countries by people added 1970 → 2022
df['Added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Added').copy()
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
n_africa = (top15['Group'] == 'Africa').sum()

fig = px.bar(
    top15, x='Added', y='Country', orientation='h',
    color='Group',
    color_discrete_map={'Africa': ORANGE, 'Other continents': GREY},
    category_orders={'Country': top15['Country'].tolist()},   
    text=(top15['Added'] / 1e6).round(0).astype(int).astype(str) + ' M',
    hover_data={'Continent': True, 'Group': False, 'Added': ':,'},
    labels={'Added': 'People added 1970–2022', 'Country': ''},
    template='plotly_white', height=600, width=950,
)
fig.update_traces(textposition='outside', marker_line_width=0)
fig.update_layout(
    title=f'{n_africa} of the 15 countries that added the most people since 1970 are in <span style="color:{ORANGE}"><b>Africa</b></span>',
    font=FONT, showlegend=False,
    xaxis=dict(visible=False, range=[0, top15['Added'].max() * 1.12]),
    yaxis=dict(showgrid=False),
)
fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a bubble chart because the question is about the relationship between two numerical variables, land area and population, and bubble size adds a third one, density. 
Both axes use a log scale because area and population differ by thousands of times; on linear axes almost all countries would be squeezed into one corner. Colour shows the continent, using a CVD-safe palette.

In [13]:
# Q3 — Bubble chart: land area vs 2022 population (size = density)
import math

d3 = df.copy()
d3['Density_size'] = d3['Density'].clip(upper=1000)   

fig = px.scatter(
    d3, x='Area', y='2022 Population',
    size='Density_size', size_max=35,
    color='Continent', color_discrete_map=continent_colours,
    log_x=True, log_y=True,
    hover_name='Country',
    hover_data={'Density_size': False, 'Density': ':,.0f', 'Area': ':,', '2022 Population': ':,'},
    labels={'Area': 'Land area (km², log scale)', '2022 Population': 'Population 2022 (log scale)',
            'Continent': ''},
    opacity=0.75,
    template='plotly_white', height=650, width=1000,
)
fig.update_traces(marker=dict(line=dict(width=0.5, color='white')))


fig.update_layout(
    title=('Bigger countries usually have more people — but Bangladesh and Singapore are far more crowded,<br>'
           'while  Canada, Australia and Mongolia are far emptier than their size suggests'),
    font=FONT,
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor=LIGHTEST_GREY),
    legend=dict(orientation='h', x=0, y=1.04),
    margin=dict(t=110, r=40),
)
fig.add_annotation(x=1, y=-0.12, xref='paper', yref='paper', xanchor='right', showarrow=False,
                   text='Bubble size = density (people per km², capped at 1,000)',
                   font=dict(size=10, color=TEXT_GREY))
fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a horizontal box plot because the question asks about spread within and across continents, not just the average. 
A box plot shows the median, the middle 50% of countries, the range and the outliers all at once, which makes continents easy to compare. Every country is also shown as a dot, so the full distribution is visible. Continents are sorted by median, highest at the top.

In [14]:
# Q4 — Box plot: spread of population density within and across continents
import math

med = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
order = med.index.tolist()                                  
world_med = df['Density'].median()

fig = px.box(
    df, x='Density', y='Continent',
    color='Continent', color_discrete_map=continent_colours,   
    category_orders={'Continent': order},
    points='all',                                         
    log_x=True,                                            
    hover_name='Country',
    hover_data={'Continent': False, 'Density': ':,.1f'},
    labels={'Density': 'People per km² (log scale)', 'Continent': ''},
    template='plotly_white', height=550, width=1000,
)
fig.update_traces(marker=dict(size=4, opacity=0.6), line=dict(width=1.5), jitter=0.4)

# Reference line: world median (the line takes the real value, the label takes log10)
fig.add_vline(x=world_med, line_dash='dash', line_color=TEXT_GREY, line_width=1.5)
fig.add_annotation(x=math.log10(world_med), y=1.02, yref='paper', showarrow=False, xanchor='left',
                   text=f'World median: {world_med:,.0f}/km²', font=dict(color=TEXT_GREY, size=11))


fig.update_layout(
    title=(f'Density varies far more within continents than between them<br>'
           f'{order[0]} has the highest median ({med.iloc[0]:,.0f}/km²), {order[-1]} the lowest ({med.iloc[-1]:,.0f}/km²)'),
    font=FONT, showlegend=False,
    xaxis=dict(gridcolor=LIGHTEST_GREY, tickvals=[0.1, 1, 10, 100, 1000, 10000],
               ticktext=['0.1', '1', '10', '100', '1,000', '10,000']),
    yaxis=dict(showgrid=False),
    margin=dict(t=100),
)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a choropleth map because the question asks where in the world populations grew or shrank, which is a geographic question.
I used a diverging colour scale centred at 0%: orange means the population shrank, white means no change, and blue means it grew. Orange and blue are safe for colour-blind viewers, unlike red and green.

In [ ]:
# Q5 — Choropleth: % population change 2000 → 2022
df['Change_pct'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100

n_shrank = (df['Change_pct'] < 0).sum()
n_shrank_eu = ((df['Change_pct'] < 0) & (df['Continent'] == 'Europe')).sum()

fig = px.choropleth(
    df, locations='CCA3', locationmode='ISO-3',
    color='Change_pct',
    color_continuous_scale=diverging_scale,
    range_color=[-50, 50],                 
    hover_name='Country',
    hover_data={'CCA3': False, 'Change_pct': ':+.1f', '2000 Population': ':,', '2022 Population': ':,'},
    labels={'Change_pct': 'Change 2000–2022 (%)'},
    projection='natural earth',
    template='plotly_white', height=550, width=1000,
)
fig.update_layout(
    title=(f'Almost the whole world grew since 2000 — {n_shrank} countries shrank, '
           f'{n_shrank_eu} of them in <span style="color:{ORANGE}"><b>Europe</b></span>'),
    font=FONT,
    geo=dict(showframe=False, showcoastlines=False, projection_rotation=dict(lon=0)),
    coloraxis_colorbar=dict(title='% change', ticksuffix='%', thickness=15, len=0.6,
                            tickvals=[-50, -25, 0, 25, 50], ticktext=['≤ −50%', '−25%', '0%', '+25%', '≥ +50%']),
    margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a treemap because the question is about continents and the countries within them, and a treemap shows both in one figure. Each rectangle’s area equals that country’s population, and each label shows its share of the world.
But it is hard to find true(correct) colours for treemap that would acceptable by everyone`s eyes.

In [9]:
# Q6 — Treemap: world population 2022 by continent and country
cont_share = df.groupby('Continent')['2022 Population'].sum() / df['2022 Population'].sum()
top_cont = cont_share.idxmax()

fig = px.treemap(
    df, path=[px.Constant('World'), 'Continent', 'Country'],
    values='2022 Population',
    color='Continent', color_discrete_map=continent_colours,
    template='plotly_white', height=700, width=1100,
)
fig.update_traces(
    texttemplate='%{label}<br>%{percentRoot:.1%}',        # share of the world
    hovertemplate='<b>%{label}</b><br>%{value:,} people<br>%{percentRoot:.1%} of world<extra></extra>',
    root_color='white',
    marker_line_color='white', marker_line_width=1,
)
# Grey out parent nodes (World + continents) so only countries carry colour
fig.data[0].marker.colors = [
    LIGHT_GREY if (node_id == 'World' or node_id.count('/') == 1) else col
    for node_id, col in zip(fig.data[0].ids, fig.data[0].marker.colors)
]
fig.update_layout(
    title=(f'{top_cont} is home to {cont_share.max():.0%} of the world\'s 8 billion people — '
           f'China and India alone hold over a third'),
    font=FONT,
    margin=dict(l=10, r=10, t=60, b=10),
)
fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a 100% stacked bar chart because the question is about shares of world population changing over time, so each bar (one per year) adds up to 100%. 

Africa is the only coloured segment (orange) and sits on the baseline, which makes its growth easy to compare from year to year. Asia, Europe and the rest of the world are in shades of grey as context.

In [ ]:
# Q7 — 100% stacked bar: share of world population, 1970 → 2022
d7 = df_long.copy()
d7['Group'] = np.where(d7['Continent'].isin(['Africa', 'Asia', 'Europe']), d7['Continent'], 'Rest of the world')
shares = d7.groupby(['Year', 'Group'])['Population'].sum().reset_index()
shares['Share'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum')
shares['Year'] = shares['Year'].astype(str)                 # years as categories
shares['Label'] = np.where(shares['Group'] == 'Africa', shares['Share'].map('{:.0%}'.format), '')

af = shares.loc[shares['Group'] == 'Africa'].set_index('Year')['Share']
eu = shares.loc[shares['Group'] == 'Europe'].set_index('Year')['Share']

fig = px.bar(
    shares, x='Year', y='Share', color='Group', text='Label',
    color_discrete_map={'Africa': ORANGE, 'Asia': GREY, 'Europe': LIGHT_GREY, 'Rest of the world': LIGHTEST_GREY},
    category_orders={'Group': ['Africa', 'Asia', 'Europe', 'Rest of the world']},   # Africa on the baseline
    hover_data={'Label': False, 'Share': ':.1%', 'Population': ':,'},
    labels={'Share': 'Share of world population', 'Year': '', 'Group': ''},
    template='plotly_white', height=550, width=950,
)
fig.update_traces(textposition='inside', insidetextanchor='start',
                  textfont=dict(color='white', size=12),
                  marker_line_color='white', marker_line_width=1)

last = shares.loc[shares['Year'] == '2022'].set_index('Group')['Share']
cum = 0
for g in ['Africa', 'Asia', 'Europe', 'Rest of the world']:
    fig.add_annotation(x=1, xref='paper', y=cum + last[g] / 2, text=g, showarrow=False, xanchor='left', xshift=6,
                       font=dict(size=11, color=ORANGE if g == 'Africa' else TEXT_GREY))
    cum += last[g]

fig.update_layout(
    title=(f'<span style="color:{ORANGE}"><b>Africa</b></span>\'s share of the world rose from '
           f'{af["1970"]:.0%} to {af["2022"]:.0%} since 1970, while Europe\'s fell from {eu["1970"]:.0%} to {eu["2022"]:.0%}'),
    font=FONT, showlegend=False, bargap=0.25,
    yaxis=dict(visible=False, range=[0, 1]),
    xaxis=dict(showgrid=False, type='category'),   # years as categories, not numbers
    margin=dict(r=130),
)
fig.show()
